# 02 — Column-by-Column Exploration

Before any cleaning, each column is examined on its own — its shape, fill rate, distribution, and a few real values — so cleaning decisions are made with the data in view rather than guessed.

Approach: convert the raw JSONL once into a flattened Parquet store (fast to reload), then walk one column at a time. For each column: look at stats and a chart, note anything that needs cleaning, decide what to do. Cleaning code is added per column as decisions are made, not up front.

The 11 source columns: `uid`, `title`, `journal`, `pubdate`, `pubdate_raw`, `pubdate_precision`, `abstract_sections`, `authors`, `mesh_terms`, `keywords`, `coi_statement`.

## 1. Setup
Resolves paths (works from `notebooks/` or the project root) and lists the raw monthly files. Uses `orjson` for faster JSON parsing.

In [ ]:
import os, re, glob
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    import orjson
    def jloads(b):
        """Parse a JSON line (orjson if available, ~2-4x faster than stdlib)."""
        return orjson.loads(b)
except ImportError:
    import json
    def jloads(b):
        return json.loads(b)


def find_project_root() -> str:
    """Locate the project root (the directory containing ``data/``).

    Walks up from the working directory so the notebook runs from ``notebooks/`` or root.

    Returns:
        str: Absolute path to the project root.
    """
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd


ROOT = find_project_root()
RAW_DATA_DIR = os.path.join(ROOT, "data", "0_raw", "results")   # input: monthly JSONL
FLAT_DIR     = os.path.join(ROOT, "data", "1_flat")             # fast store: flattened Parquet
os.makedirs(FLAT_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "results_*.jsonl")))
assert files, f"no results_*.jsonl in {RAW_DATA_DIR}"
print(f"root: {ROOT}\ninput files: {len(files)}")

## 2. Build the Parquet store (run once)
Converts the raw JSONL into a flattened, columnar Parquet store one time, streaming in batches so memory stays flat. Nested fields (abstract, authors, MeSH, keywords) are flattened and helper columns derived. No rows are dropped and no text is altered beyond joining the abstract sections — it is a faithful, fast-loading mirror of the raw data. Re-running rebuilds it; skip this cell on later sessions once the store exists.

In [ ]:
FLAT_SCHEMA = pa.schema([
    ("uid", pa.string()), ("title", pa.string()), ("journal", pa.string()),
    ("year", pa.int16()), ("pubdate", pa.string()), ("pubdate_precision", pa.string()),
    ("abstract", pa.string()), ("abstract_len", pa.int32()),
    ("author_names", pa.list_(pa.string())), ("affiliations", pa.list_(pa.string())), ("n_authors", pa.int32()),
    ("mesh_descriptors", pa.list_(pa.string())), ("n_mesh", pa.int32()),
    ("keywords", pa.list_(pa.string())), ("n_keywords", pa.int32()),
    ("coi_statement", pa.string()), ("has_coi", pa.bool_()), ("source_month", pa.string()),
])


def flatten_abstract(sections) -> str:
    """Join structured abstract sections into one string (labels prefixed).

    Args:
        sections (list): The raw ``abstract_sections`` value.

    Returns:
        str: The abstract text, or ``""`` if empty/invalid.
    """
    if not isinstance(sections, list):
        return ""
    out = []
    for sec in sections:
        if isinstance(sec, dict) and sec.get("text"):
            label = (sec.get("label") or "").strip()
            out.append(f"{label}: {sec['text']}" if label else sec["text"])
    return " ".join(out)


def build_flat_store(batch_size: int = 100_000) -> int:
    """Convert the raw JSONL once into a flattened Parquet store for fast exploration.

    Streams every record, flattens the nested fields (abstract, authors, MeSH, keywords)
    and derives helper columns, writing Parquet shards in batches so memory stays flat.
    No rows are dropped and no text is altered beyond joining the abstract sections, so
    the store is a faithful, fast-loading mirror of the raw data.

    Args:
        batch_size (int): Rows accumulated before each Parquet shard is written.

    Returns:
        int: Number of shards written to ``FLAT_DIR``.
    """
    buf, idx = [], 0

    def flush(buffer, index):
        if not buffer:
            return index
        pq.write_table(pa.Table.from_pylist(buffer, schema=FLAT_SCHEMA),
                       os.path.join(FLAT_DIR, f"flat_{index:05d}.parquet"), compression="zstd")
        return index + 1

    for fp in tqdm(files, desc="JSONL -> Parquet"):
        m = re.search(r"results_(\d{4})_(\d{2})", fp)
        month = f"{m.group(1)}-{m.group(2)}" if m else ""
        with open(fp, "rb") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                r = jloads(line)
                authors = r.get("authors") or []
                names = [a.get("name", "") for a in authors if isinstance(a, dict) and a.get("name")]
                affs = [x for a in authors if isinstance(a, dict) for x in (a.get("affiliations") or [])]
                mesh = [t.get("descriptor", "") for t in (r.get("mesh_terms") or [])
                        if isinstance(t, dict) and t.get("descriptor")]
                kws = [k for k in (r.get("keywords") or []) if k]
                coi = (r.get("coi_statement") or "").strip()
                abstract = flatten_abstract(r.get("abstract_sections"))
                pd_str = r.get("pubdate", "")
                buf.append({
                    "uid": str(r.get("uid", "")), "title": r.get("title", ""),
                    "journal": r.get("journal", ""),
                    "year": int(pd_str[:4]) if pd_str[:4].isdigit() else None,
                    "pubdate": pd_str, "pubdate_precision": r.get("pubdate_precision", ""),
                    "abstract": abstract, "abstract_len": len(abstract),
                    "author_names": names, "affiliations": affs, "n_authors": len(names),
                    "mesh_descriptors": mesh, "n_mesh": len(mesh),
                    "keywords": kws, "n_keywords": len(kws),
                    "coi_statement": coi, "has_coi": bool(coi), "source_month": month,
                })
                if len(buf) >= batch_size:
                    idx = flush(buf, idx); buf = []
    idx = flush(buf, idx)
    return idx


n_shards = build_flat_store()
print(f"wrote {n_shards} Parquet shard(s) to {FLAT_DIR}")

 stage 1: the same data flattened into Parquet (nested fields unpacked, fast to load, nothing dropped)

## 3. Load
Loads the Parquet store into a DataFrame. The whole corpus loads in seconds; for a single statistic, read only the columns needed (near-instant, minimal memory) — see the commented example.

In [ ]:
# Whole corpus (seconds to load; a few GB in memory):
df = pd.read_parquet(FLAT_DIR)

# For a single statistic, read only the columns needed (near-instant, tiny memory), e.g.:
#   pd.read_parquet(FLAT_DIR, columns=["year", "abstract_len"])

print(f"loaded {len(df):,} rows  |  columns: {list(df.columns)}")
df.head(3)

## 4. Data Inspection


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import collections
import pandas as pd
sns.set_theme(style="whitegrid")

### abstract

In [ ]:
print(df['abstract_len'].describe().astype(int))
print("empty:", int((df['abstract_len'] == 0).sum()))
print("very short (<100):", int((df['abstract_len'] < 100).sum()))
plt.figure(figsize=(9, 3))
sns.histplot(df['abstract_len'].clip(upper=4000), bins=60, color='#8064a2')
plt.title('abstract length (chars, clipped 4000)'); plt.xlabel('chars'); plt.tight_layout(); plt.show()
df.loc[df['abstract_len'].between(1, 80), 'abstract'].head(10).tolist()

### title

In [ ]:
tl = df['title'].fillna('').str.len()
print(tl.describe().astype(int))
print("empty:", int((df['title'].fillna('').str.strip() == '').sum()))
print("bracketed [...]:", int(df['title'].fillna('').str.startswith('[').sum()))
plt.figure(figsize=(9, 3))
sns.histplot(tl.clip(upper=300), bins=60, color='#4472c4')
plt.title('title length (chars)'); plt.xlabel('chars'); plt.tight_layout(); plt.show()
df.loc[df['title'].fillna('').str.startswith('['), 'title'].head(5).tolist()

### journal

In [ ]:
print("distinct journals:", df['journal'].nunique())
print("empty:", int((df['journal'].fillna('').str.strip() == '').sum()))
top = df['journal'].value_counts().head(20)
plt.figure(figsize=(9, 6))
sns.barplot(x=top.values, y=top.index, color='#4472c4')
plt.title('top 20 journals'); plt.xlabel('records'); plt.tight_layout(); plt.show()

### year / pubdate_precision

In [ ]:
print(df['pubdate_precision'].value_counts(dropna=False))
print("\nyear range:", int(df['year'].min()), "-", int(df['year'].max()))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
prec = df['pubdate_precision'].value_counts()
sns.barplot(x=prec.index, y=prec.values, ax=ax[0], color='#9bbb59'); ax[0].set_title('precision')
yr = df['year'].value_counts().sort_index()
sns.lineplot(x=yr.index, y=yr.values, ax=ax[1], color='#4472c4'); ax[1].set_title('records per year')
plt.tight_layout(); plt.show()

### authors / affiliations

In [ ]:
print("no-author rows:", int((df['n_authors'] == 0).sum()))
print(df['n_authors'].describe().astype(int))
plt.figure(figsize=(9, 3))
sns.histplot(df['n_authors'].clip(upper=30), bins=30, color='#4472c4')
plt.title('authors per paper (clipped 30)'); plt.xlabel('authors'); plt.tight_layout(); plt.show()
empty_aff = (df['affiliations'].str.len() == 0)
print("empty affiliations:", int(empty_aff.sum()), f"({empty_aff.mean()*100:.1f}%)")
df.loc[~empty_aff, 'affiliations'].head(3).tolist()

### mesh_descriptors

In [ ]:
print("rows with no MeSH:", int((df['n_mesh'] == 0).sum()))
mc = collections.Counter(d for lst in df['mesh_descriptors'] for d in lst)
print("distinct descriptors:", len(mc))
top = pd.Series(dict(mc.most_common(15)))
plt.figure(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, color='#9bbb59')
plt.title('top 15 MeSH descriptors'); plt.xlabel('occurrences'); plt.tight_layout(); plt.show()
cov = df.assign(has_mesh=df['n_mesh'] > 0).groupby('year')['has_mesh'].mean()
plt.figure(figsize=(9, 3))
sns.lineplot(x=cov.index, y=cov.values, color='#c0504d')
plt.title('share with MeSH, by year'); plt.tight_layout(); plt.show()

### keywords

In [ ]:
has_kw = df['n_keywords'] > 0
print("rows with keywords:", int(has_kw.sum()), f"({has_kw.mean()*100:.1f}%)")
cov = df.assign(has_kw=has_kw).groupby('year')['has_kw'].mean()
plt.figure(figsize=(9, 3))
sns.lineplot(x=cov.index, y=cov.values, color='#4472c4')
plt.title('share with keywords, by year'); plt.tight_layout(); plt.show()
df.loc[has_kw, 'keywords'].head(5).tolist()

### coi_statement / has_coi

In [ ]:
print("with COI:", int(df['has_coi'].sum()), f"({df['has_coi'].mean()*100:.1f}%)")
cov = df.groupby('year')['has_coi'].mean()
plt.figure(figsize=(9, 3))
sns.lineplot(x=cov.index, y=cov.values, color='#9bbb59')
plt.title('share with COI statement, by year'); plt.tight_layout(); plt.show()
df.loc[df['has_coi'], 'coi_statement'].head(3).tolist()

### uid

In [ ]:
print("rows:", len(df))
print("unique uids:", df['uid'].nunique())
print("duplicate uids:", int(df['uid'].duplicated().sum()))
print("non-numeric uids:", int((~df['uid'].astype(str).str.fullmatch(r'\d+')).sum()))

## 5. Final cleaning of data

With every column inspected, the corpus is now deduplicated and the few unusable rows removed, producing the final clean dataset. Each step below was confirmed by the diagnostics above before being applied — nothing is dropped on assumption.

### 5.1 The duplicate PMIDs

The flat store holds 4,259,464 rows but only 3,082,363 unique PMIDs. The gap of 1,177,101 is records that appear in more than one month's file — every duplicated PMID has exactly two copies.

This is not corruption. PubMed's `[PDAT]` query can return the same article for more than one month, through two mechanisms confirmed in the diagnostics:

- **Imprecise dates (the bulk).** Roughly 79% of the corpus has year-only or year-month precision. A record dated only "2009" is returned by the `[PDAT]` query for several months of 2009, so the harvest saved it once per month-bucket it matched. The year-gap between a duplicate's two buckets is 0–1 years for almost all cases, which is exactly this effect.
- **Dual print/electronic dates (a small tail).** A few records carry both an old print date and a much later online date (e.g. a 1998 article digitized in 2023), so `[PDAT]` matches both — a decades-wide gap between buckets. This is the long tail in the year-gap histogram.

Crucially, the two copies of every duplicated PMID are identical: 0 differ in `pubdate`, 0 in `pubdate_precision`, 0 in `title`, and 100% have identical abstracts. The only thing that differs is which month-query fetched the record (`source_month`). Deduplicating by `uid` therefore loses no information — and keeping the first (earliest `source_month`) keeps the original publication date for the print/electronic cases.

This also reconciles a number from the start of the project: 3,082,363 unique PMIDs matches the ~3.1M returned by the single all-time query. That all-time count was the number of distinct articles; the 4.26M is the sum across months with imprecise records multiply-counted. After deduplication the two figures agree.

In [ ]:
# every duplicated PMID's copies must agree on the fields that matter
dups = df[df['uid'].duplicated(keep=False)]
chk = dups.groupby('uid').agg(
    n_pubdate=('pubdate', 'nunique'),
    n_precision=('pubdate_precision', 'nunique'),
    n_title=('title', 'nunique'),
    n_abstract=('abstract', 'nunique'),
)
print(f"duplicated PMIDs:           {len(chk):,}")
print(f"  differing pubdate:        {int((chk['n_pubdate']   > 1).sum()):,}")
print(f"  differing precision:      {int((chk['n_precision'] > 1).sum()):,}")
print(f"  differing title:          {int((chk['n_title']     > 1).sum()):,}")
print(f"  differing abstract:       {int((chk['n_abstract']  > 1).sum()):,}")
if (chk[['n_pubdate','n_precision','n_title','n_abstract']] > 1).any().any():
    print("\n-> some copies disagree; inspect before deduping")
else:
    print("\n-> all copies identical in key fields; dedup by uid (keep first) is safe")

In [ ]:
# every duplicated PMID's copies must agree on the fields that matter
dups = df[df['uid'].duplicated(keep=False)]
chk = dups.groupby('uid').agg(
    n_pubdate=('pubdate', 'nunique'),
    n_precision=('pubdate_precision', 'nunique'),
    n_title=('title', 'nunique'),
    n_abstract=('abstract', 'nunique'),
)
print(f"duplicated PMIDs:                 {len(chk):,}")
print(f"  with differing pubdate:         {int((chk['n_pubdate']   > 1).sum()):,}")
print(f"  with differing precision:       {int((chk['n_precision'] > 1).sum()):,}")
print(f"  with differing title:           {int((chk['n_title']     > 1).sum()):,}")
print(f"  with differing abstract:        {int((chk['n_abstract']  > 1).sum()):,}")
if (chk[['n_pubdate','n_precision','n_title','n_abstract']] > 1).any().any():
    print("\n-> some copies disagree; inspect before deduping")
else:
    print("\n-> all copies identical in key fields; dedup by uid (keep first) is safe")

### 5.2 Where the duplicates fall

Two charts make the duplication concrete: rows per year split into the copies that are kept versus removed, and the year-gap between the two buckets each duplicated PMID lands in (which shows the two mechanisms — a dominant 0–1 year spike and a thin decades-wide tail).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

removed = df['uid'].duplicated(keep='first')          # True = this row is dropped in dedup
piv = (df.assign(state=removed.map({False: 'kept', True: 'removed'}))
         .groupby(['year', 'state']).size().unstack(fill_value=0)
         .reindex(columns=['kept', 'removed']))
piv.plot(kind='bar', stacked=True, figsize=(14, 4), color=['#4472c4', '#c0504d'])
plt.title('rows per year: kept (unique after dedup) vs removed (duplicate copy)')
plt.xlabel('year'); plt.ylabel('rows'); plt.tight_layout(); plt.show()
print("kept (unique):", int((~removed).sum()), "| removed:", int(removed.sum()))

dups = df[df['uid'].duplicated(keep=False)].copy()
dups['sm_year'] = dups['source_month'].str.slice(0, 4).astype(int)
gap = dups.groupby('uid')['sm_year'].agg(lambda s: s.max() - s.min())
plt.figure(figsize=(10, 3))
sns.histplot(gap, bins=40, color='#8064a2')
plt.title('year-gap between the two buckets a duplicated PMID lands in')
plt.xlabel('years apart'); plt.ylabel('PMIDs'); plt.tight_layout(); plt.show()

### 5.3 Deduplicate by PMID

Collapse to one row per PMID, keeping the first occurrence (the earliest `source_month`). This reduces the corpus from ~4.26M rows to ~3.08M unique articles.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset='uid', keep='first').copy()   # keep = earliest source_month
print(f"{before:,} -> {len(df):,}  (removed {before-len(df):,} duplicate-PMID rows)")
print("unique PMIDs now:", df['uid'].nunique(), "(should equal row count)")

### 5.4 Drop empty abstracts

22 records have an empty abstract — unusable for text mining — and are removed. The 1,732 very short (under 100 characters) abstracts are kept: inspection showed they are real, if minimal, abstracts rather than junk.

In [ ]:
before = len(df)
df = df[df['abstract_len'] > 0].copy()
print(f"{before:,} -> {len(df):,}  (removed {before-len(df):,} empty-abstract rows)")

### 5.5 Date boundary

The pubdate year spans 1994–2026. The handful of 2026 dates and the still-accruing final months of 2025 are the live edge of PubMed, where counts are not yet stable or reproducible. For a frozen, reproducible dataset the range can be trimmed (for example to `year <= 2025`); for the fullest coverage the recent months can be kept and labelled as partial. Set this deliberately before saving.

In [ ]:
# Decide the boundary. To trim the live edge to a frozen range, uncomment:
# before = len(df)
# df = df[df['year'] <= 2025].copy()
# print(f"{before:,} -> {len(df):,}  (trimmed to year <= 2025)")
print("year range:", int(df['year'].min()), "-", int(df['year'].max()))

## 6. Save the clean dataset

The deduplicated, cleaned corpus is written to `data/2_clean/` as Parquet shards. This is the usable dataset that downstream analysis and the publishable export build on.

In [ ]:
import os, pyarrow as pa, pyarrow.parquet as pq

CLEAN_DIR = os.path.join(ROOT, "data", "2_clean")
os.makedirs(CLEAN_DIR, exist_ok=True)
SHARD = 200_000
n = (len(df) + SHARD - 1) // SHARD
for i in range(n):
    part = df.iloc[i*SHARD:(i+1)*SHARD]
    pq.write_table(pa.Table.from_pandas(part, preserve_index=False),
                   os.path.join(CLEAN_DIR, f"clean_{i:05d}.parquet"), compression="zstd")
print(f"saved {n} shard(s) to {CLEAN_DIR}  ({len(df):,} rows, {len(df.columns)} columns)")
df.head(3)

## 7.Export

Writes a Kaggle-ready copy of the clean dataset for each policy in `EXPORT_POLICIES`, each into its own folder under `data/2_export/`, so several variants can be produced and compared in one run. The policy governs only the author-written text (`abstract`, `coi_statement`); all bibliographic metadata is always kept.

- `"none"` — metadata + derived fields only; no abstract or COI text. Safest for redistribution: bibliographic facts are not copyrightable, and abstracts are re-fetchable from the PMIDs with notebook 00.
- `"oa_only"` — abstract text only for PMIDs in the PMC commercial-use open-access subset (CC0 / CC BY family); all other records get metadata only.
- `"all"` — all abstract text. Highest reuse value, but abstracts may be under publisher copyright; redistribute only with appropriate license terms.

Every row carries an `abstract_redistributable` flag recording the decision.

### Which variant is published

All three variants are produced locally for comparison, but the variant uploaded to Kaggle is `"none"` — metadata only, no abstract text. The reasoning:

1. **It is the legally safe choice.** Bibliographic metadata (PMIDs, titles, journals, authors, affiliations, MeSH, dates) is supplied by NLM without restriction and can be redistributed freely. Abstract text, in bulk, may be under publisher or author copyright; NLM does not hold that copyright and places responsibility on whoever redistributes it. The `"none"` variant avoids that exposure entirely.
2. **It loses little in practice.** The abstracts remain fully recoverable — anyone can re-fetch them from the PMIDs using notebook 00 — so the dataset stays complete and useful without redistributing copyrighted text.
3. **The derived fields carry the value.** The flattened authors, affiliations, MeSH descriptors, keywords, and COI flags are the distinctive, freely publishable parts.

The `"oa_only"` and `"all"` variants are built here only for local inspection. `"oa_only"` is restricted to the PMC commercial-use subset (permissively licensed), so its abstracts would in principle be redistributable, but it covers only a minority of records and is kept local for now. `"all"` carries real copyright exposure and is not published. An abstract-bearing release can follow later if licensing is cleared.

Kaggle settings for the published `"none"` variant: license **"Other (specified in description)"** (not CC0 — the metadata is NLM-sourced, not the uploader's to relicense), the attribution line *"Courtesy of the U.S. National Library of Medicine,"* and a snapshot/currency disclaimer noting the data is a static extract that does not reflect the most current PubMed records.

### 7.1 Download the open-access list (for `oa_only` only)

Downloads NCBI's commercial-use open-access file list, which identifies PubMed Central articles under permissive (CC0 / CC BY family) licenses. Needed only to build the `oa_only` variant; skip it entirely if publishing metadata-only. As of mid-2026 NCBI serves this list from the `deprecated/` path.

In [ ]:
import os, urllib.request

oa_dir = os.path.join(ROOT, "data", "oa")
os.makedirs(oa_dir, exist_ok=True)

url = "https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_comm_use_file_list.csv"   # ~602 MB
dest = os.path.join(oa_dir, "oa_comm_use_file_list.csv")
if os.path.exists(dest):
    print("already have", os.path.basename(dest), f"({os.path.getsize(dest)/1e6:.0f} MB)")
else:
    print("downloading", os.path.basename(dest), "(~602 MB)...")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
        print("  saved", dest, f"({os.path.getsize(dest)/1e6:.0f} MB)")
    except Exception as e:
        if os.path.exists(dest):
            os.remove(dest)
        print(f"  FAILED: {e}  (download via browser into data/oa/ if this persists)")

### 7.2 Tag the open-access PMIDs

Reads the commercial-use list and builds `OA_PMID_SET` — the PMIDs whose abstracts are permissively licensed. The list has a `PMID` column directly, so no PMC-ID join is needed. The license breakdown is printed as a sanity check. Skip this (leave `RUN_OA_TAGGING = False`) if not building `oa_only`.

In [ ]:
import os
import pandas as pd

RUN_OA_TAGGING = True          # set False to skip oa_only entirely
OA_PMID_SET = set()

if RUN_OA_TAGGING:
    oa_dir = os.path.join(ROOT, "data", "oa")
    oa = pd.read_csv(os.path.join(oa_dir, "oa_comm_use_file_list.csv"),
                     usecols=["PMID", "License"], dtype={"PMID": "string"})
    print("license breakdown:")
    print(oa["License"].value_counts(dropna=False))
    oa = oa.dropna(subset=["PMID"])
    oa = oa[oa["PMID"].str.strip() != ""]
    OA_PMID_SET = set(oa["PMID"].str.strip())
    print(f"\nopen-access (commercial-use) PMIDs: {len(OA_PMID_SET):,}")
else:
    print("OA tagging skipped (RUN_OA_TAGGING = False).")

### 7.3 Export all variants

Produces one folder per policy under `data/2_export/`. `abstracts_none/` is the variant uploaded to Kaggle; `abstracts_oa_only/` and `abstracts_all/` are local. A progress bar tracks each variant; `oa_only` is skipped automatically if `OA_PMID_SET` is empty.

In [ ]:
import os, glob, collections
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import pyarrow.compute as pc
from tqdm.auto import tqdm


def find_project_root() -> str:
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd


ROOT       = find_project_root()
CLEAN_DIR  = os.path.join(ROOT, "data", "2_clean")
EXPORT_DIR = os.path.join(ROOT, "data", "2_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

EXPORT_POLICIES = ["none", "oa_only", "all"]      # only "none" is uploaded
TEXT_COLS = ["abstract", "coi_statement"]
MAX_YEAR  = 2025                                  # drop the 2026 live-edge records

# load the OA set from disk if the commercial-use file is present; else oa_only is skipped
OA_PMID_SET = set()
_oa_file = os.path.join(ROOT, "data", "oa", "oa_comm_use_file_list.csv")
if os.path.exists(_oa_file):
    OA_PMID_SET = set(pd.read_csv(_oa_file, usecols=["PMID"], dtype={"PMID": "string"})
                        .dropna()["PMID"].str.strip())
    print(f"OA PMIDs loaded: {len(OA_PMID_SET):,}")
else:
    print("no OA file at data/oa/oa_comm_use_file_list.csv -> oa_only will be skipped")

clean_shards = sorted(glob.glob(os.path.join(CLEAN_DIR, "clean_*.parquet")))
assert clean_shards, f"no clean shards in {CLEAN_DIR} — run the section-5 save first"


def export_policy(policy: str) -> collections.Counter:
    """Write a publishable copy of the clean corpus under one abstract policy, trimmed to MAX_YEAR.

    Args:
        policy (str): "none", "oa_only", or "all".

    Returns:
        collections.Counter: {"with_abstract", "metadata_only"} row counts.
    """
    out_dir = os.path.join(EXPORT_DIR, f"abstracts_{policy}")
    os.makedirs(out_dir, exist_ok=True)
    tally = collections.Counter()
    for i, sp in enumerate(tqdm(clean_shards, desc=f"export {policy}")):
        t = pq.read_table(sp)
        t = t.filter(pc.less_equal(t["year"], MAX_YEAR))
        if t.num_rows == 0:
            continue
        cols = {name: col.to_pylist() for name, col in zip(t.schema.names, t.columns)}
        flags = []
        for j in range(t.num_rows):
            if policy == "all":
                ok = True
            elif policy == "oa_only":
                ok = cols["uid"][j] in OA_PMID_SET
            else:
                ok = False
            flags.append(ok)
            if not ok:
                for c in TEXT_COLS:
                    cols[c][j] = ""
            tally["with_abstract" if ok else "metadata_only"] += 1
        cols["abstract_redistributable"] = flags
        pq.write_table(pa.table(cols), os.path.join(out_dir, f"part_{i:05d}.parquet"), compression="zstd")
    return tally


for policy in EXPORT_POLICIES:
    if policy == "oa_only" and not OA_PMID_SET:
        print(f"[skip] policy 'oa_only' needs the OA file (OA_PMID_SET is empty)")
        continue
    t = export_policy(policy)
    total = t["with_abstract"] + t["metadata_only"]
    folder = os.path.join(EXPORT_DIR, f"abstracts_{policy}")
    print(f"policy '{policy}': {total:,} rows (<= {MAX_YEAR})  "
          f"with_abstract={t['with_abstract']:,}  metadata_only={t['metadata_only']:,}  ->  {folder}")

In [ ]:
import os, glob
import pandas as pd


def find_project_root() -> str:
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd


ROOT = find_project_root()
SRC = os.path.join(ROOT, "data", "2_export", "abstracts_none")   # the exact data uploaded
shards = sorted(glob.glob(os.path.join(SRC, "*.parquet")))
assert shards, f"no parquet in {SRC} — run the export (Block 1) first"
df = pd.read_parquet(SRC)


def list_fill(col):
    """Percent of rows where a list column is non-empty."""
    return df[col].map(lambda v: len(v) > 0).mean() * 100


prec = df["pubdate_precision"].value_counts()
imprecise = df["pubdate_precision"].isin(["year", "year_month"]).mean() * 100

print("=== numbers for the Kaggle description ===")
print(f"records:            {len(df):,}")
print(f"unique PMIDs:       {df['uid'].nunique():,}")
print(f"year range:         {int(df['year'].min())}–{int(df['year'].max())}")
print(f"distinct journals:  {df['journal'].nunique():,}")
print(f"top journals:       {list(df['journal'].value_counts().head(5).index)}")
print(f"authors/paper:      mean {df['n_authors'].mean():.1f}, median {int(df['n_authors'].median())}, max {int(df['n_authors'].max())}")
print(f"avg abstract length: {df['abstract_len'].mean():.0f} chars (text not included; length retained)")
print()
print("date precision:")
for k, v in prec.items():
    print(f"  {k:12} {v:,}  ({v/len(df)*100:.0f}%)")
print(f"  imprecise (year / year_month): {imprecise:.0f}%")
print()
print("field coverage:")
print(f"  affiliations present: {list_fill('affiliations'):.0f}%")
print(f"  MeSH present:         {list_fill('mesh_descriptors'):.0f}%")
print(f"  keywords present:     {list_fill('keywords'):.0f}%")
print(f"  COI statement:        {df['has_coi'].mean()*100:.0f}%")

In [ ]:
print(check)